imports

In [2]:
from pathlib import Path
import random
import shutil

setting up input path and output paths

In [3]:
RAW_DIR = Path('../dataset/raw')
OUTPUT_DIR = Path('../dataset')

*

In [4]:
TRAIN = 0.70
VALIDATION = 0.20
TEST = 0.10

SEED = 42
random.seed(SEED)

image_extensions = ['.jpg', '.jpeg', '.png']

*

In [1]:
for class_folder in RAW_DIR.iterdir():
    if not class_folder.is_dir():
        continue

    class_name = class_folder.name

    images = [
        file for file in class_folder.iterdir() 
        if file.suffix.lower() in image_extensions
    ]

    random.shuffle(images)

    total_images = len(images)
    train_count = int(total_images * TRAIN)
    valid_count = int(total_images * VALIDATION)

    train_images = images[:train_count]
    valid_images = images[train_count:train_count + valid_count]
    test_images = images[train_count + valid_count:]    

    for split, split_images in {
        'TRAIN': train_images,
        'VALIDATION': valid_images,
        'TEST': test_images
    }.items():
        split_class_dir = OUTPUT_DIR / split.lower() / class_name

        # if users create the folder manually, or if the script is run multiple times, we want to avoid errors and just overwrite the files
        if split_class_dir.exists():
            # print(f'Warning: {split_class_dir} already exists. Files may be overwritten.')
            shutil.rmtree(split_class_dir)

        split_class_dir.mkdir(parents=True, exist_ok=True)

        for image_path in split_images:
            # print(f'Copying {image_path} to {split_class_dir}')
            shutil.copy(
                image_path, 
                split_class_dir / image_path.name
            )
    print(f'{class_name}: {len(train_images)} train, {len(valid_images)} valid, {len(test_images)} test')

NameError: name 'RAW_DIR' is not defined

*

In [10]:
import pandas as pd

base_dir = Path("../dataset")
splits = ["train", "validation", "test"]
image_extensions = [".jpg", ".jpeg", ".png"]

rows = []

for split in splits:
    for class_folder in sorted((base_dir / split).iterdir()):
        if class_folder.is_dir():
            count = sum(
                1 for file in class_folder.iterdir()
                if file.suffix.lower() in image_extensions
            )

            rows.append({
                "Split": split,
                "Class": class_folder.name,
                "Image Count": count
            })

split_df = pd.DataFrame(rows)
split_df

,Split,Class,Image Count
0,train,Tomato__Bacterial_Spot,1488
1,train,Tomato__Early_Blight,700
2,train,Tomato__Healthy,1113
3,train,Tomato__Late_Blight,1336
4,train,Tomato__Leaf_Mold,666
5,train,Tomato__Mosaic_Virus,261
6,train,Tomato__Septoria_Leaf_Spot,1239
7,train,Tomato__Target_Spot,982
8,train,Tomato__Two_Spotted_Spider_Mites,1173
9,train,Tomato__YellowLeaf_Curl_Virus,2245


summary

In [11]:
split_summary = split_df.groupby("Split")["Image Count"].sum().reset_index()
split_summary

,Split,Image Count
0,test,1610
1,train,11203
2,validation,3198
